# Phase 3 - Ground Truth / Change Labels
Delhi NCR Urban Change Intelligence (2022 -> 2026)

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

import sys, os
REPO = '/content/delhi-ncr-satellite-change'
if not os.path.exists(REPO):
    !git clone https://github.com/karan02566-prog/delhi-ncr-satellite-change.git {REPO}
else:
    !cd {REPO} && git pull
if REPO not in sys.path:
    sys.path.append(REPO)


In [ ]:
import rasterio
import numpy as np
from src.data.label_generation import (
    compute_change_labels, label_patches, select_verification_sample,
    export_verification_chips, geographic_split, summarize_split, save_split_manifest,
)

DRIVE_DIR = '/content/drive/MyDrive/delhi_ncr_change_detection'
T1_PATH = f'{DRIVE_DIR}/delhi_ncr_t1_2022_masked.tif'
T2_PATH = f'{DRIVE_DIR}/delhi_ncr_t2_2026_masked.tif'

with rasterio.open(T1_PATH) as src:
    t1_array = src.read()
with rasterio.open(T2_PATH) as src:
    t2_array = src.read()

print('T1 shape:', t1_array.shape, 'T2 shape:', t2_array.shape)


In [ ]:
result = compute_change_labels(t1_array, t2_array)
print(result.thresholds)


In [ ]:
PATCH_SIZE = 256  # must match whatever patch size Phase 5 training will use

patches = label_patches(result.label, patch_size=PATCH_SIZE, min_valid_fraction=0.5)
print(f'Usable patches (>=50% valid pixels): {len(patches)}')


In [ ]:
verification_sample = select_verification_sample(patches, n_per_stratum=15)
print(f'Verification sample size: {len(verification_sample)}')

export_verification_chips(
    t1_array, t2_array, verification_sample,
    out_dir=f'{DRIVE_DIR}/reports/label_verification',
    patch_size=PATCH_SIZE,
)
print('Chips + verification_log.csv written to Drive. Manually review before trusting labels.')


In [ ]:
splits = geographic_split(patches, raster_width=t1_array.shape[2], patch_size=PATCH_SIZE)
print(summarize_split(splits))

save_split_manifest(splits, result.thresholds, out_path=f'{DRIVE_DIR}/data_labels_split_manifest.json')
print('Split manifest saved.')
